# Compare & Sweep

Two questions this answers:

1. **Which strategy is better?** — train several on one universe and tabulate.
2. **Which settings are better?** — sweep a hyperparameter grid for one strategy.

Both pin the universe once and hand the identical ticker list to every job. A
comparison table whose rows were fitted on different draws from the cache
measures the draws as much as the models, and nothing in the numbers tells you
which is which.

## 1. One universe for every job

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from portfolio_agent.lab import Lab

lab = Lab(universe_size=40, name="comparison")
lab.save_universe("universe/comparison.json")
lab

## 2. Compare strategies

`save=False` by default: a comparison should not replace the models in
`models/`. Pass `save=True` when you want the winners written out — each entry
gets its own checkpoint name so they cannot overwrite one another.

In [ ]:
report = lab.compare(["india_sac"], epochs=20)
report.to_frame()

In [ ]:
best = report.best()
print("best:", best.strategy, best.trainer, best.artifact.primary_metric())
print("universe:", report.universe.fingerprint, f"({len(report.universe)} tickers)")

if report.failures:
    print("failed:", report.failures)

## 3. Sweep hyperparameters

The cross product of the grid, one job per point. A failing point is recorded
and the sweep continues — losing nine good results because the tenth had a bad
setting is the worse outcome.

In [ ]:
sweep_report = lab.sweep(
    "india_sac",
    {"gamma": [0.0, 0.9], "entropy_coef": [0.05, 0.2]},
    epochs=20,
)
sweep_report.to_frame()

In [ ]:
frame = sweep_report.to_frame()
ok = frame[frame["status"] == "ok"]
if not ok.empty and "metric" in ok:
    ax = ok.plot.barh(x="label", y="metric", figsize=(9, 4), legend=False)
    ax.set_xlabel("validation Sortino")

## 4. The same thing from the command line

Everything above has a CLI equivalent — the notebook is for exploring, the CLI
for repeatable runs:

```bash
portfolio-agent list-trainers --name sac

portfolio-agent train --strategy india_sac \
    --set epochs=200 --set gamma=0.0 \
    --save-snapshot universe/comparison.json

portfolio-agent train-bulk --strategies india_sac \
    --sweep gamma=0.0,0.9 --sweep entropy_coef=0.05,0.2 \
    --universe-snapshot universe/comparison.json
```

Passing `--universe-snapshot` to a later run reproduces this comparison on the
same names, which is what makes two sessions' numbers comparable at all.